# Spike D — Fine-tune timing on free-tier GPU

**Goal:** fine-tune the chosen base model (F5 OR ZipVoice-Distill, whichever
Spike A + E point us at) on ~10 min of reference audio. Measure:

1. Wall-clock to reasonable convergence
2. Whether it fits a single free-tier session (Kaggle 9h, Colab free 12h)
3. Number of session restarts required for longer runs
4. Subjective quality on held-out sentences (3–5 samples for listen)

**This is the longest spike.** Run it overnight. If it doesn't converge in
< 12 h on a T4, we must either (a) use Colab Pro paid tier, (b) switch to
a smaller base model, or (c) accept that training requires a paid runner.

## What "reasonable convergence" means

- Loss curve flattening (not necessarily at the theoretical minimum)
- Validation MCD (mel-cepstral distortion) vs the teacher < 5.0
- Samples sound *recognizably like the reference speaker* to a human ear

We're not trying to ship; we're trying to prove the free-tier timing box
fits. Going from "sounds right" to "production quality" is Phase 2 scope.

## 0. Prep reference audio

Drop a clean-speech reference clip into `/content/reference.wav` and its
transcript into `/content/reference.txt` before running the notebook.

- **Length:** 10–20 min of a single speaker
- **Quality:** no background music, no reverb, no overlapping voices
- **Format:** WAV 24 kHz mono preferred; MP3 fine (we'll resample)
- **Transcript:** sentence-aligned with the audio is ideal. If you only
  have the audio, the preprocessing cell below will use whisper-timestamped
  large-v3 to produce one.

In [ ]:
import os, sys, json, time, subprocess
REF_AUDIO = '/content/reference.wav'
REF_TEXT  = '/content/reference.txt'
CKPT_DIR  = '/content/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

assert os.path.exists(REF_AUDIO), f'Upload your reference audio to {REF_AUDIO}'
print('Reference:', REF_AUDIO, os.path.getsize(REF_AUDIO) / 1024 / 1024, 'MB')
if os.path.exists(REF_TEXT):
    print('Transcript provided.')
else:
    print('No transcript — will run whisper-timestamped below.')

In [ ]:
!pip install -q torch==2.4.0 torchaudio==2.4.0
!pip install -q f5-tts==1.0.3 || pip install -q git+https://github.com/SWivid/F5-TTS.git
!pip install -q zipvoice || pip install -q git+https://github.com/k2-fsa/ZipVoice.git
!pip install -q whisper-timestamped==1.15.4
print('Deps installed.')

## 1. Preprocessing

Resample → loudness-normalize → segment → force-align.

In [ ]:
import torchaudio

wav, sr = torchaudio.load(REF_AUDIO)
if sr != 24000:
    wav = torchaudio.functional.resample(wav, sr, 24000)
    sr = 24000
if wav.shape[0] > 1:
    wav = wav.mean(dim=0, keepdim=True)  # downmix to mono
torchaudio.save('/content/reference_24k.wav', wav, sr)
print(f'Resampled → /content/reference_24k.wav  ({wav.shape[1] / sr / 60:.1f} min)')

In [ ]:
# Auto-transcribe if needed.
if not os.path.exists(REF_TEXT):
    import whisper_timestamped as wt
    model_w = wt.load_model('large-v3', device='cuda')
    result = wt.transcribe(model_w, '/content/reference_24k.wav',
                            language='en', beam_size=5, vad='auditok')
    with open(REF_TEXT, 'w') as f:
        for seg in result['segments']:
            f.write(f"{seg['start']:.2f}\t{seg['end']:.2f}\t{seg['text'].strip()}\n")
    print(f'Wrote {len(result["segments"])} segments → {REF_TEXT}')

## 2. Training run

This cell invokes the model-specific fine-tune script. Pick ONE of the
two paths below; comment out the other. We expect to be here for 6–12 h.

In [ ]:
# Shared timing wrapper — reads checkpoints every N steps and logs wall
# clock + estimated time-to-next-milestone. Works against whichever
# underlying trainer writes to CKPT_DIR on its own schedule.
import threading, glob

TIMING_LOG = '/content/timing.jsonl'
stop_event = threading.Event()
start_ts = time.time()

def timing_watcher():
    last_ckpts = set()
    while not stop_event.is_set():
        now = time.time()
        ckpts = set(glob.glob(f'{CKPT_DIR}/*.pt')) | set(glob.glob(f'{CKPT_DIR}/*.safetensors'))
        new = ckpts - last_ckpts
        for p in sorted(new):
            rec = {
                'ts': now,
                'elapsed_s': now - start_ts,
                'elapsed_h': (now - start_ts) / 3600,
                'checkpoint': os.path.basename(p),
                'size_mb': os.path.getsize(p) / 1024 / 1024,
            }
            with open(TIMING_LOG, 'a') as f:
                f.write(json.dumps(rec) + '\n')
            print(f"[{rec['elapsed_h']:.2f} h] {rec['checkpoint']} ({rec['size_mb']:.0f} MB)")
        last_ckpts = ckpts
        time.sleep(30)

watcher = threading.Thread(target=timing_watcher, daemon=True)
watcher.start()
print('Timing watcher started — will log new checkpoints.')

In [ ]:
# ==== Path A: F5-TTS fine-tune ====
# F5 uses a train.sh-style config + `accelerate launch`. Mirror what
# the Voice Studio Deep Clone tab already renders into train.sh.
#
# Typical hyperparams for a single-voice fine-tune on 10 min of audio:
#   - steps: 8000
#   - batch_size: 2 with grad_accum=8
#   - lr: 2e-5
#   - ckpt every 500 steps
#   - warmup: 500
#
# Uncomment this block to run Path A.
#
# !accelerate launch --num_processes 1 \
#   -m f5_tts.train.finetune_cli \
#   --exp_name F5TTS_v1_Base \
#   --dataset_name my_voice \
#   --epochs 1 --max_steps 8000 --save_per_updates 500 \
#   --learning_rate 2e-5 --batch_size_per_gpu 2 --grad_accumulation_steps 8 \
#   --tokenizer pinyin --pretrain /content/F5TTS_v1_Base/model_1250000.pt \
#   --output_dir {CKPT_DIR}
print('Path A (F5) is commented out. Uncomment to run.')

In [ ]:
# ==== Path B: ZipVoice-Distill fine-tune ====
# ZipVoice has first-party fine-tune tooling. Pull their README at run-time
# for the current CLI (interface has drifted in 2025).
#
# Uncomment to run Path B.
#
# !python -m zipvoice.bin.train \
#   --model-name zipvoice_distill \
#   --manifest /content/manifest.json \
#   --num-steps 5000 \
#   --save-every 500 \
#   --lr 1e-5 \
#   --output-dir {CKPT_DIR}
print('Path B (ZipVoice) is commented out. Uncomment to run.')

## 3. Resume-from-checkpoint smoke test

Kaggle kicks you at 9h, Colab free at 12h. Training must survive session
restarts. Simulate it: kill the kernel mid-run, re-run from the top, and
verify the trainer picks up the latest checkpoint.

In [ ]:
# Verify checkpoint files exist and are loadable.
import glob, torch
ckpts = sorted(glob.glob(f'{CKPT_DIR}/*.pt') + glob.glob(f'{CKPT_DIR}/*.safetensors'))
print(f'{len(ckpts)} checkpoints:')
for c in ckpts[-5:]:
    print(' ', os.path.basename(c), f'{os.path.getsize(c) / 1024 / 1024:.0f} MB')

# Spot-check the latest loads without error.
if ckpts:
    latest = ckpts[-1]
    try:
        sd = torch.load(latest, map_location='cpu', weights_only=True)
        print(f'Loaded {os.path.basename(latest)} — {len(sd)} tensors')
    except Exception as e:
        print(f'Load failed: {e}')

## 4. Synthesize held-out samples

Generate 3 sentences the model hasn't seen:
1. Neutral tone
2. Question intonation
3. Longer compound sentence

These get saved to `/content/samples/` for subjective listen.

In [ ]:
os.makedirs('/content/samples', exist_ok=True)

HELDOUT = [
    'The morning light filtered through the kitchen blinds.',
    'Have you already finished the report you were working on last night?',
    'Although the conference had run long, nobody wanted to skip the final panel, which promised to be the most interesting of the day.',
]

# Path-specific inference here. Skipped in this stub; copy the matching
# `infer_cli` invocation from the README of whichever base model was trained.
for i, text in enumerate(HELDOUT):
    print(f'TODO: synthesize sample {i} → /content/samples/sample_{i}.wav')
    print(f'  "{text}"')

## 5. Final timing report

In [ ]:
stop_event.set()
total_h = (time.time() - start_ts) / 3600

report = {
    'spike': 'D',
    'base_model': 'TODO: f5 or zipvoice',
    'reference_audio_min': wav.shape[1] / sr / 60,
    'total_wall_clock_h': total_h,
    'fits_kaggle_free_9h': total_h < 9,
    'fits_colab_free_12h': total_h < 12,
    'n_checkpoints': len(ckpts) if 'ckpts' in dir() else 0,
    'final_loss': 'TODO: read from trainer log',
    'validation_mcd': 'TODO: run post-training validate.py',
    'subjective_notes': 'TODO: listen to /content/samples/ and note in REPORT.md',
}

report_path = '/content/timing_report.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2, default=str)
print(json.dumps(report, indent=2, default=str))
print()
print(f'Total: {total_h:.2f} h')
print(f'Kaggle free (9h):  {"FITS" if total_h < 9 else "DOES NOT FIT — need restarts"}')
print(f'Colab free (12h):  {"FITS" if total_h < 12 else "DOES NOT FIT — need Colab Pro"}')

In [ ]:
try:
    from google.colab import files
    files.download(report_path)
    files.download(TIMING_LOG)
    # Optionally download the latest checkpoint for offline inspection
    # files.download(ckpts[-1]) if ckpts else None
except ImportError:
    print('Download manually from /content/')

## 6. What to record in REPORT.md

Spike D section of `spikes/REPORT.md`:
- Reference audio length
- Total wall clock on T4
- Fits 9h Kaggle? Fits 12h Colab free?
- # session restarts required (0 if fits single session)
- Final loss + MCD
- Subjective quality notes on the 3 held-out samples

### Decision tree for the go/no-go

- **Fits 9h Kaggle:** 🟢 GREEN — free-tier is viable; Phase 1 runner set
  becomes simple.
- **Fits 12h Colab free:** 🟡 YELLOW — viable but needs Colab, not Kaggle.
  Keep both runners; let user pick.
- **Fits Colab Pro 24h:** 🟡 YELLOW — user opts into $10/mo.
- **Doesn't fit 24h single session:** 🟠 ORANGE — Modal / Lightning or
  multi-session training with external checkpoint storage. Plan's
  backend-agnostic design already accommodates this, but it's a UX hit.
- **Doesn't converge at all after 24h:** 🔴 RED — replan, likely swap
  base model or reduce training steps.